In [0]:
%pip install us
import us
import polars as pl
from pyspark.sql.types import *

### Create Delta Lake Tables
 *Simple table creation - similar to CTAS (Create Table as Select)*

In [0]:
df = spark.read.option('header', 'true').csv('dbfs:/Volumes/geography/staging/raw_data/us-cities-top-1k-multi-year.csv')
write_options = {
    'path': 'r2://mvcostanzo-geography@4f3914487748b814b84bb931230b53e8.r2.cloudflarestorage.com/unmanaged/us-city-data'
}
df.write.format('delta').partitionBy('year').options(**write_options).saveAsTable('geography.staging.us_city_data')

*Table definition then load*

In [0]:
#Table first, then load

stateTableSchema = StructType(
    [
        StructField('stateName', StringType()),
        StructField('stateAbbr', StringType())
    ]
)
spark.catalog.createTable(
    tableName='geography.staging.states', 
    source =  'delta', 
    schema=stateTableSchema)

In [0]:
%sql
CREATE OR REPLACE TABLE geography.staging.states2
(
  stateName STRING,
  stateAbbr STRING
)

In [0]:
stateNameList = [x.name for x in us.STATES]
stateAbbrList = [x.abbr for x in us.STATES]

df = spark.createDataFrame(zip(stateNameList, stateAbbrList), ['stateName', 'stateAbbr'])
df.write.saveAsTable('geography.staging.states', mode='append')

### Query Delta Tables

In [0]:
%sql
SELECT * FROM  geography.staging.us_city_data WHERE State = 'North Carolina' ORDER BY City, Year;

In [0]:
%sql
SELECT * FROM geography.staging.states WHERE stateName LIKE 'North%'

In [0]:
%sql
SELECT 
  cd.City
  ,cd.State
  ,cd.Population
  ,cd.Year
  ,s.stateAbbr   
FROM geography.staging.us_city_data AS cd 
JOIN geography.staging.states  as s 
ON (s.stateName = cd.State)   
WHERE s.stateAbbr = 'NC' AND cd.Year = 2014
ORDER BY cd.Population DESC;

### Append New Data

In [0]:
%sql
SELECT * FROM geography.staging.states WHERE stateAbbr = 'DC';

In [0]:
new_row_df = spark.createDataFrame([{
    'stateName': 'District of Columbia',
    'stateAbbr': 'DC'
}])
new_row_df.write.saveAsTable('geography.staging.states', mode='append')


In [0]:
%sql
SELECT * FROM geography.staging.states WHERE stateAbbr = 'DC';

### Update Table
*Merge Table*

In [0]:
%sql
SELECT * FROM geography.staging.states WHERE stateName LIKE 'North%'

In [0]:
%sql
UPDATE geography.staging.states 
SET stateName = 'North Cakalacky' 
WHERE stateAbbr= 'NC'

In [0]:
%sql
SELECT * FROM geography.staging.states WHERE stateName LIKE 'North%'

### Time Travel 

In [0]:
%sql
DESCRIBE HISTORY geography.staging.states;

In [0]:
%sql
SELECT * FROM geography.staging.states VERSION AS OF 1

In [0]:
df = spark.read.table('geography.staging.states').toPandas()
print(df.sort_values(by='stateName', ignore_index=True))

### Update Table
*Full Overwrite*

In [0]:
df = spark.read.table('geography.staging.states')
df = df.replace("North Cakalacky", "Superior Carolina", "stateName")
df.write.saveAsTable('geography.staging.states', mode='overwrite')


In [0]:
%sql
SELECT * FROM geography.staging.states WHERE stateName LIKE 'North%'

In [0]:
%sql
SELECT * FROM geography.staging.states WHERE stateName LIKE '%Carolina'

### Vacuum

In [0]:
%sql
SELECT * FROM geography.staging.states VERSION AS OF 1

In [0]:
%sql
SELECT * FROM geography.staging.states VERSION AS OF 3 ORDER BY stateName

In [0]:
%sql
ALTER TABLE geography.staging.states SET TBLPROPERTIES ('delta.deletedFileRetentionDuration' = '0 hours');

In [0]:
%sql
DESCRIBE HISTORY geography.staging.states;

In [0]:
%sql
DESCRIBE DETAIL geography.staging.states;

In [0]:
%sql
VACUUM geography.staging.states FULL DRY RUN;

In [0]:
%sql
VACUUM geography.staging.states FULL;

In [0]:
%sql
DESCRIBE HISTORY geography.staging.states;

In [0]:
%sql
SELECT * FROM geography.staging.states VERSION AS OF 3 ORDER BY stateName

In [0]:
%sql
SELECT * FROM geography.staging.states ORDER BY stateName

### Optimize

In [0]:
%sql
INSERT INTO geography.staging.states
(stateName, stateAbbr)
VALUES ('Puerto Rico', 'PR');

In [0]:
%sql
OPTIMIZE geography.staging.states;

In [0]:
%sql
VACUUM geography.staging.states FULL;